# Parte 2 · Optimización vectorial de distancias espaciales

## Contexto y objetivo
Un script calcula la distancia de **1 000 000 de puntos 2D al origen (0, 0)** mediante un bucle. Tu misión es optimizarlo, medir el antes y el después, y generar una gráfica de los datos procesados.

Esta versión sigue el código de la nueva diapositiva: dos vectores `x` e `y` con coordenadas uniformes en `[0, 100)`. Sustituye el planteamiento anterior de puntos 3D. El bucle original está resuelto; la optimización y la gráfica quedan para ti.

## Datos y contrato
- `x` e `y`: arreglos NumPy de una dimensión, tipo `float64`, de igual longitud N.
- El punto i es `(x[i], y[i])`. Su distancia es `sqrt(x[i]**2 + y[i]**2)`.
- Devuelve un arreglo `float64` de forma `(N,)`, en el mismo orden y sin modificar las coordenadas.
- No se piden distancias entre pares de puntos.
- Usa los mismos `x` e `y` para ambas versiones. No cambies la generación de datos del experimento principal.

## 1. Optimización
Implementa `distancias_vectorizadas(x, y)` mediante operaciones de arreglos. Elimina los bucles explícitos: no uses `for`, `while`, comprensiones, `map` ni `np.vectorize`. Para practicar la traducción de la fórmula, usa operaciones elementales; no delegues la distancia a una función que ya la calcule.

## 2. Cronometría y validación
Implementa `comparar_tiempos(x, y, repeticiones=5)`:
1. Ejecuta un calentamiento de cada función fuera de la medición.
2. Usa `time.perf_counter()` para medir cada versión al menos cinco veces; alterna cuál se ejecuta primero.
3. Incluye la reserva del resultado y los temporales de cada función. Excluye la generación, las verificaciones, las impresiones y las gráficas.
4. Compara todas las distancias mediante `np.testing.assert_allclose`, con `rtol=1e-12` y `atol=1e-12`. Comprueba también forma, tipo y que las entradas no cambien.
5. Reporta las medianas y la aceleración `T_for / T_vectorizado`.
6. Devuelve `(distancias, resumen)`, donde `distancias` es la salida vectorizada y `resumen` es un diccionario con las claves `tiempo_for_s`, `tiempo_vectorizado_s` y `aceleracion`.

Los tiempos de 2 a 5 segundos de la diapositiva son orientativos, no una condición que deba cumplirse en tu equipo. La igualdad se comprueba con tolerancia por el redondeo de punto flotante; explica además la equivalencia matemática de ambas expresiones.

## 3. Visualización
Implementa `graficar_resultados(x, y, distancias)` con Matplotlib y devuelve la figura:
- Panel A: dispersión 2D, coloreada según la distancia al origen. Incluye etiquetas, título, barra de color y la misma escala en ambos ejes.
- Para que la gráfica sea legible y ágil, muestra como máximo 10 000 puntos de una muestra reproducible. Usa los mismos índices para `x`, `y` y `distancias`.
- Panel B: histograma de las distancias de **todos** los puntos. Etiqueta distancia y frecuencia.
- La muestra solo afecta a la dispersión: el cálculo y la validación usan el millón completo.

## Pistas
- Traduce la fórmula escalar a operaciones sobre los dos vectores completos.
- Observa qué trabajo ya realizan las operaciones de NumPy sin iterar en Python.
- La distancia mínima posible es 0 y el límite superior es `100 * sqrt(2)`.
- Vectorización no equivale necesariamente a ejecución en varios núcleos. No presupongas que este cálculo elemental usa BLAS o LAPACK.

## Continuación en MATLAB
Más adelante reproduce el experimento con `x` e `y` como vectores fila, una salida preasignada para el `for`, y medición con `tic/toc` o `timeit`. Distingue operaciones matriciales y elemento a elemento. Para comparar lenguajes utiliza las mismas coordenadas exportadas; una misma semilla no garantiza iguales datos en Python y MATLAB. Por ahora solo se entregan la plantilla Python y el notebook.

## Entrega
Código optimizado, validación completa, tabla de tiempos antes/después, gráfica y explicación breve del cambio y su aceleración. No hay solución vectorizada incluida.

## Ejecución
Necesitas NumPy y Matplotlib: `python3 -m pip install numpy matplotlib`.
En Jupyter puedes instalarlos con `%pip install numpy matplotlib`.
Ejecuta primero el código base. Completa las tres funciones pendientes antes de ejecutar las secciones de solución; mientras estén pendientes producirán `NotImplementedError`.

## Preparación y datos

In [ ]:
import time
import statistics
import numpy as np

def generar_datos(n=1_000_000, semilla=42):
    """Equivalente al esquema rand(1, N) * 100 de la diapositiva."""
    rng = np.random.default_rng(semilla)
    x = rng.random(n) * 100
    y = rng.random(n) * 100
    return x, y

## Código base proporcionado
Conserva esta función: es el antes de la optimización.

In [ ]:
def distancias_for(x, y):
    """Código base: conserva esta función para comparar el antes y después."""
    N = len(x)
    distancias = np.zeros(N, dtype=np.float64)
    for i in range(N):
        distancias[i] = np.sqrt(x[i]**2 + y[i]**2)
    return distancias

In [ ]:
N = 1_000_000
x, y = generar_datos(N)
inicio = time.perf_counter()
distancias_base = distancias_for(x, y)
tiempo_base = time.perf_counter() - inicio
print(f"Bucle original: {tiempo_base:.6f} s")
print("Primeras cinco distancias:", distancias_base[:5])

## Tu solución vectorizada

In [ ]:
def distancias_vectorizadas(x, y):
    """Aplica Pitágoras elemento a elemento, sin ciclos de Python."""
    return np.sqrt(x**2 + y**2)


## Casos pequeños
Se comprueban ambas implementaciones.

In [ ]:
def comprobar_casos(funcion):
    """Casos pequeños conocidos; no sustituyen la validación del millón."""
    casos = [
        ([0, 3, 6, 0, 0.3], [0, 4, 8, 5, 0.4], [0, 5, 10, 5, 0.5]),
        ([5], [12], [13]),
        ([], [], []),
    ]
    for valores_x, valores_y, esperado in casos:
        x = np.array(valores_x, dtype=np.float64)
        y = np.array(valores_y, dtype=np.float64)
        original_x, original_y = x.copy(), y.copy()
        resultado = funcion(x, y)
        if not isinstance(resultado, np.ndarray):
            raise AssertionError("Devuelve un arreglo NumPy.")
        if resultado.shape != x.shape or resultado.dtype != np.float64:
            raise AssertionError("Revisa la forma (N,) y el tipo float64.")
        np.testing.assert_allclose(resultado, esperado, rtol=1e-12, atol=1e-12)
        np.testing.assert_array_equal(x, original_x)
        np.testing.assert_array_equal(y, original_y)
    print(f"{funcion.__name__}: casos pequeños correctos.")

In [ ]:
comprobar_casos(distancias_for)
comprobar_casos(distancias_vectorizadas)

## Tu comparación de tiempos y resultados

In [ ]:
def comparar_tiempos(x, y, repeticiones=5):
    if not isinstance(repeticiones, int) or repeticiones < 5:
        raise ValueError("Se requieren al menos cinco repeticiones.")
    original_x, original_y = x.copy(), y.copy()
    funciones = (distancias_for, distancias_vectorizadas)
    tiempos = [[], []]

    def validar(resultado):
        assert isinstance(resultado, np.ndarray)
        assert resultado.shape == x.shape
        assert resultado.dtype == np.float64
        np.testing.assert_array_equal(x, original_x)
        np.testing.assert_array_equal(y, original_y)

    # Calentamiento y referencia fuera del cronómetro.
    referencia = distancias_for(x, y)
    validar(referencia)
    calentamiento = distancias_vectorizadas(x, y)
    validar(calentamiento)
    np.testing.assert_allclose(calentamiento, referencia, rtol=1e-12, atol=1e-12)
    del calentamiento

    for repeticion in range(repeticiones):
        orden = (0, 1) if repeticion % 2 == 0 else (1, 0)
        for indice in orden:
            inicio = time.perf_counter()
            resultado = funciones[indice](x, y)
            duracion = time.perf_counter() - inicio
            tiempos[indice].append(duracion)
            # Validaciones fuera del intervalo medido; se revisa todo el arreglo.
            validar(resultado)
            np.testing.assert_allclose(resultado, referencia, rtol=1e-12, atol=1e-12)
            if indice == 1:
                distancias = resultado
            del resultado

    tiempo_for = statistics.median(tiempos[0])
    tiempo_vectorizado = statistics.median(tiempos[1])
    return distancias, {
        "tiempo_for_s": tiempo_for,
        "tiempo_vectorizado_s": tiempo_vectorizado,
        "aceleracion": tiempo_for / tiempo_vectorizado,
    }


In [ ]:
distancias, resumen = comparar_tiempos(x, y, repeticiones=5)
print(resumen)

## Tu gráfica de los datos procesados

In [ ]:
def graficar_resultados(x, y, distancias):
    import matplotlib.pyplot as plt
    rng = np.random.default_rng(2026)
    indices = rng.choice(len(x), size=min(10_000, len(x)), replace=False)
    figura, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
    puntos = ax1.scatter(x[indices], y[indices], c=distancias[indices],
                         s=5, cmap="viridis", alpha=0.75)
    ax1.set(xlabel="Coordenada x", ylabel="Coordenada y",
            title="A · Distancia al origen (muestra de hasta 10 000)")
    ax1.set_aspect("equal", adjustable="box")
    figura.colorbar(puntos, ax=ax1, label="Distancia al origen")
    ax2.hist(distancias, bins=60, color="#337eaa", edgecolor="white", linewidth=0.3)
    ax2.set(xlabel="Distancia al origen", ylabel="Frecuencia (número de puntos)",
            title="B · Distancias de todos los puntos")
    return figura


In [ ]:
figura = graficar_resultados(x, y, distancias)
import matplotlib.pyplot as plt
plt.show()

# Solución: distancias al origen con ciclos y vectores

## 1. Qué se calcula
Se generan 1 000 000 de puntos `(x[i], y[i])`, con coordenadas uniformes en `[0, 100)`, mediante `np.random.default_rng(42)`, tal como indica la plantilla. Se calcula la distancia de cada punto al origen, no la distancia entre pares de puntos.

Por el teorema de Pitágoras:

`d[i] = sqrt(x[i]² + y[i]²)`.

Por ejemplo, para `(3, 4)`, la distancia es `sqrt(9 + 16) = 5`. El resultado contiene un millón de distancias, en el mismo orden que los puntos originales.

## 2. Python con ciclos
```python
def distancias_for(x, y):
    N = len(x)
    distancias = np.zeros(N, dtype=np.float64)
    for i in range(N):
        distancias[i] = np.sqrt(x[i]**2 + y[i]**2)
    return distancias
```
Primero se reserva el arreglo de resultados. Después, el ciclo recorre las posiciones de 0 a N−1, obtiene las dos coordenadas, eleva ambas al cuadrado, suma y calcula la raíz cuadrada. Se conserva exactamente el código base.

## 3. Python con vectores
```python
def distancias_vectorizadas(x, y):
    return np.sqrt(x**2 + y**2)
```
`x**2` eleva cada coordenada x al cuadrado; `y**2` hace lo mismo con y. La suma combina elementos en la misma posición y `np.sqrt` calcula sus raíces. En la posición i se obtiene exactamente la misma fórmula matemática que con el ciclo.

La diferencia está en cómo se ejecuta: NumPy procesa los arreglos mediante rutinas compiladas y evita un millón de iteraciones gestionadas por Python. No se usa `np.vectorize` ni una función que ya calcule distancias. Esta vectorización no implica necesariamente varios núcleos ni requiere atribuir el cálculo a BLAS o LAPACK.

Ambas versiones requieren tiempo O(N) y un resultado de tamaño O(N). La expresión vectorizada crea arreglos temporales para cuadrados y suma. Cada arreglo float64 de un millón de elementos ocupa 8 MB (aproximadamente 7.63 MiB); no todos los temporales tienen que coexistir. La velocidad se obtiene a cambio de ese trabajo y memoria temporal adicionales.

## 4. Medición y validación
Se usa el mismo par de arreglos en ambas versiones. Primero se ejecuta una vez cada función para calentar. Después se realizan cinco mediciones por versión con `time.perf_counter()`, alternando el orden: ciclo/vector, vector/ciclo, etc.

El cronómetro incluye la reserva del resultado y los temporales dentro de cada función. La generación de datos, comprobaciones, impresiones y gráficas quedan fuera. Se usa la mediana para reducir la influencia de mediciones aisladas lentas.

Se comprobaron los casos `(0,0)`, `(3,4)`, `(6,8)`, `(0,5)`, `(0.3,0.4)`, `(5,12)` y los arreglos vacíos. Además, todas las mediciones del millón completo se validaron con `np.testing.assert_allclose`, `rtol=1e-12` y `atol=1e-12`. Se revisaron forma `(N,)`, tipo float64 y conservación de las entradas.

La comparación tolera diferencias pequeñas por redondeo: `abs(resultado - referencia) <= atol + rtol * abs(referencia)`.

| N | Mediana con ciclo (s) | Mediana vectorizada (s) | Aceleración |
|---:|---:|---:|---:|
| 1 000 000 | 0.384715 | 0.001283 | 299.85× |

La aceleración es `T_ciclo / T_vectorizado`. Estos son tiempos reales de esta ejecución, no valores garantizados para otros equipos.

Entorno: Python 3.12.14, NumPy 2.5.4, Matplotlib 3.11.2; macOS-26.6.2-arm64-arm-64bit, arquitectura arm64.

## 5. Gráfica
El panel A representa una muestra reproducible de 10 000 puntos. Se seleccionan los mismos índices para x, y y distancia. La escala de ambos ejes es igual para evitar deformar la geometría. Los puntos próximos al origen tienen menor distancia y los cercanos a `(100,100)` tienen mayor distancia.

El panel B usa las distancias de los 1 000 000 de puntos, no solo la muestra. Las frecuencias no son uniformes: distintas franjas de distancia ocupan distintas áreas dentro del cuadrado. Las distancias están entre 0 y un límite superior no incluido de `100*sqrt(2) ≈ 141.421`.

![Gráfica](distancias.png)

## 6. Pasarlo a MATLAB
En MATLAB los índices empiezan en 1. Se utilizan vectores fila y se preasigna el resultado del ciclo.

```matlab
function distancias = distancias_for(x, y)
    N = numel(x);
    distancias = zeros(1, N, 'double');
    for i = 1:N
        distancias(i) = sqrt(x(i)^2 + y(i)^2);
    end
end

function distancias = distancias_vectorizadas(x, y)
    distancias = sqrt(x.^2 + y.^2);
end
```

En el ciclo, `x(i)` es un escalar y puede usarse `^2`. En la versión vectorizada se necesita `.^2`, que eleva cada elemento al cuadrado; `^2` representa una potencia matricial. De igual forma, `.*` y `./` serían las operaciones elemento a elemento para multiplicación y división. La suma `+` ya opera elemento a elemento para vectores del mismo tamaño y `sqrt` también.

`solucion_matlab.m` importa `coordenadas.csv`, convierte sus columnas en vectores fila, comprueba casos pequeños y el millón completo, calienta ambos métodos y mide cinco repeticiones alternadas con `tic/toc`. Finalmente muestra la tabla, genera la gráfica y guarda `resultados_matlab.csv` y `distancias_matlab.png`.

El CSV conserva 17 cifras significativas para transferir los mismos float64. Usar la misma semilla en ambos lenguajes no garantiza las mismas coordenadas. La muestra para dibujar puede ser diferente entre lenguajes; cada muestra es reproducible y el cálculo siempre usa todos los puntos.

No se ejecutó MATLAB en este entorno, por lo que sus tiempos quedan por medir. Su compilador JIT puede optimizar mucho el ciclo: no debe suponerse que tendrá la misma aceleración que Python. Para comparar lenguajes, ejecútalos en el mismo equipo, con las mismas coordenadas y condiciones similares.

## 7. Cómo ejecutar
- Python: instala `numpy` y `matplotlib` con `python3 -m pip install numpy matplotlib`; luego ejecuta `python3 solucion_python.py`. También puedes abrir `solucion_python.ipynb` y ejecutar sus celdas en orden.
- MATLAB: coloca `solucion_matlab.m` y `coordenadas.csv` en la misma carpeta, abre el archivo `.m` y pulsa Run. El script localiza el CSV junto a sí mismo.

El script Python regenera la gráfica, el CSV y el archivo de resultados; la exportación queda fuera de los tiempos medidos.


In [ ]:
# Exportación para MATLAB, fuera de la medición.
np.savetxt("coordenadas.csv", np.column_stack((x, y)), delimiter=",",
           header="x,y", comments="", fmt="%.17g")
